# Episode 1 — JAX as a Functional Array Accelerator

**Instructor notebook** · run top-to-bottom before recording.

Why JAX is not NumPy, and why that matters for ML.

| | |
|---|---|
| **Chapter** | 1.1 · Part I — Pure JAX |
| **Prereq** | Python, basic NumPy |
| **Next** | Episode 2 — JIT, tracing, and the jaxpr |

**JAX docs:** [JAX 101](https://jax.readthedocs.io/en/latest/jax-101.html) · [`jax.numpy`](https://jax.readthedocs.io/en/latest/jax.numpy.html) · [Pseudorandom numbers](https://docs.jax.dev/en/latest/random-numbers.html) · [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) · [Async dispatch](https://jax.readthedocs.io/en/latest/async_dispatch.html)


In [ ]:
import time

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np

## Pure functions and immutability

JAX has no hidden global state. Arrays are **immutable** — updates return new arrays via `.at[...].set(...)`. This purity is what makes `jit`, `grad`, and `vmap` composable later.

In [ ]:
x = jnp.array([1.0, 2.0, 3.0])

try:
    x[0] = 99.0
except TypeError as e:
    print(type(e).__name__, ":", e)

x_new = x.at[0].set(99.0)
print("original:", x)
print("updated: ", x_new)

## PRNG keys

JAX has **no hidden global RNG**. Create a **key**, **`split`** before every random draw, and never reuse a consumed subkey.

In [ ]:
key = jr.key(0)
key, subkey = jr.split(key)

a = jr.normal(subkey, (3,))
print(a)

# Same root key → same draws
key = jr.key(0)
key, subkey = jr.split(key)
b = jr.normal(subkey, (3,))
print(jnp.allclose(a, b))

## `jnp` vs NumPy — same API, different execution

NumPy runs eagerly on the host. JAX builds a deferred computation graph (we compile it in Episode 2). Start with the same matmul in both.

In [ ]:
M, K, N = 1024, 512, 1024

a_np = np.random.randn(M, K).astype(np.float32)
b_np = np.random.randn(K, N).astype(np.float32)

# NumPy — eager on CPU
t0 = time.perf_counter()
c_np = a_np @ b_np
t_np = time.perf_counter() - t0
print(f"NumPy matmul: {t_np * 1000:.2f} ms, shape {c_np.shape}")

In [ ]:
key = jax.random.key(0)
key, k_a, k_b = jax.random.split(key, 3)
a_jnp = jax.random.normal(k_a, (M, K))
b_jnp = jax.random.normal(k_b, (K, N))

# JAX — dispatches asynchronously; block to measure
t0 = time.perf_counter()
c_jnp = a_jnp @ b_jnp
jax.block_until_ready(c_jnp)
t_jnp = time.perf_counter() - t0
print(f"JAX matmul:   {t_jnp * 1000:.2f} ms, shape {c_jnp.shape}")
print("match (same problem size):", np.allclose(np.array(c_jnp), np.array(a_jnp @ b_jnp), atol=1e-4))

## Device placement

Arrays live on a **device** (CPU, GPU, TPU). Use `jax.devices()` to see what's available; `jax.device_put` moves data explicitly.

In [ ]:
print("devices:", jax.devices())
print("default backend:", jax.default_backend())

x_host = jnp.ones((4, 4))
x_device = jax.device_put(x_host, jax.devices()[0])
print("array device:", x_device.devices())

## Asynchronous dispatch

JAX returns before work finishes on accelerator hardware. Always call `jax.block_until_ready(...)` before stopping a timer.

In [ ]:
def bench(fn, *args, repeat=5):
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        out = fn(*args)
        times.append(time.perf_counter() - t0)
    return min(times) * 1000


def matmul_no_block(a, b):
    return a @ b


def matmul_block(a, b):
    return jax.block_until_ready(a @ b)


a = jax.random.normal(jax.random.key(1), (2048, 2048))
b = jax.random.normal(jax.random.key(2), (2048, 2048))

raw_ms = bench(matmul_no_block, a, b)
blocked_ms = bench(matmul_block, a, b)
print(f"without block_until_ready: {raw_ms:.2f} ms  (may under-report on GPU)")
print(f"with    block_until_ready: {blocked_ms:.2f} ms")

## Broadcasting

JAX follows [NumPy broadcasting rules](https://numpy.org/doc/stable/user/basics.broadcasting.html): compare shapes from the **right**; dimensions match if equal or one is `1`.

For a batched matmul `x @ w` with `x` shape `(B, D_in)`, `w` shape `(D_in, D_out)`, the result is `(B, D_out)`. A bias `b` with shape `(D_out,)` broadcasts across the batch rows — no extra storage.

In [ ]:
B, d_in, d_out = 64, 10, 10
key = jr.key(5)
key, k_x = jr.split(key)
x = jr.normal(k_x, (B, d_in))
w = jr.normal(jr.key(6), (d_in, d_out)) * 0.1
b = jnp.zeros(d_out)

logits = x @ w
print("broadcast_shapes:", jnp.broadcast_shapes(logits.shape, b.shape))
y = logits + b
print("y.shape:", y.shape)
print("matches row-wise add:", jnp.allclose(y, logits + b[None, :]))

## Deferred execution preview — shape-sensitive functions

JAX traces functions on **shapes and dtypes**. Calling the same Python function with different array shapes can re-trace (Episode 2 covers this in depth). Here, a side-effect counter shows retracing.

In [ ]:
trace_count = 0


def shape_logger(x):
    global trace_count
    trace_count += 1
    print(f"  Python body ran (trace #{trace_count}), x.shape = {x.shape}")
    return jnp.sum(x ** 2)


x_small = jnp.ones((8,))
x_large = jnp.ones((16,))

print("call 1:")
y1 = shape_logger(x_small)
print("call 2 (same shape):")
y2 = shape_logger(x_small)
print("call 3 (new shape):")
y3 = shape_logger(x_large)
print(f"Python body ran {trace_count} times — new shapes re-run Python (until we jit in Ep 2)")

> **Key insight:** The purity constraint is not a limitation — it is what makes compilation, differentiation, and parallelism composable. Every later chapter depends on this.

---
## Exercise

1. Time NumPy vs JAX matmul on your machine (adjust `M, K, N` if needed).
2. `split` a key and draw twice from the same root — verify reproducibility.
3. Print `jax.devices()` and the device of one `device_put` array.
4. Show `(B, D_out) + (D_out,)` broadcasting with `jnp.broadcast_shapes`.
5. Benchmark one large matmul with and without `block_until_ready`.

*(Solution below.)*

In [ ]:
# Demo answers — students replicate the cells above with their own timings.
print("devices:", [str(d) for d in jax.devices()])
print("backend:", jax.default_backend())

**Next:** Episode 2 — JIT, tracing, and the jaxpr.